# Quick start: A Flash in the Dark

This notebook is a first look at the toy detector and its data. It simulates a small data set, saves it and reads it
back, and shows what is stored: the true event parameters, the detector response, the photon bookkeeping, the
optional hits and photon paths, and the input of the neural networks of the next stage of the course.

Run it in the environment of the [README](../README.md) (`python -m pip install -e ".[analysis]"`), started with
`python -m jupyter lab` from the repository root; on JupyterHub or in VS Code, choose the kernel registered for this
environment (see "Notebooks" in the README). It takes seconds on a CPU and writes one small file to the repository's
`data/` folder, which is `../data/` from this notebook's folder: every path in the course is relative to the
repository root. The physics is described in
[docs/physics.md](../docs/physics.md), the file layout in [docs/data_format.md](../docs/data_format.md), and the tasks
are in [TASKS.md](../TASKS.md).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import prob_reco_sbi
from prob_reco_sbi.flash import (
    PRESETS,
    Interaction,
    describe,
    generate,
    load,
    network_input,
    preset,
    resolve_device,
    save,
)

print("prob_reco_sbi", prob_reco_sbi.__version__, "| torch", torch.__version__, "| device:", resolve_device("auto"))

## Configuration

A configuration (`SimConfig`) has four parts: the geometry, the optics, the readout and the prior. The four presets
form a physics ladder, each one switching on one more optical process, and every parameter can be changed on top of a
preset.

In [ ]:
for name, cfg in PRESETS.items():
    o = cfg.optics
    print(f"{name:<15s} absorption={o.absorption!s:<6s} scattering={o.scattering!s:<6s} reflection={o.reflection}")

config = preset("full")
print()
print(config.summary())

In [ ]:
custom = preset("straight", reflection=True, n_sensors=16, sensor_width=0.2)
print("reflection:", custom.optics.reflection, "| sensors:", custom.geometry.n_sensors)
print("photocoverage:", round(custom.geometry.photocoverage, 3))
custom.to_dict()["optics"]

## Simulate

`generate` draws the event parameters from the prior and tracks every photon until it is detected, absorbed or lost.
With the default `device="auto"` it runs on a GPU if PyTorch sees one. `save_hits=True` keeps every hit, and
`save_tracks=2` keeps the paths of the first `track_photons` photons of the first two events.

In [ ]:
data = generate(2000, config=config, seed=1, save_hits=True, save_tracks=2, track_photons=50)
data

In [ ]:
keys = ("seed", "device", "n_chunks", "package_version", "torch_version", "wall_time_s")
{key: data.provenance[key] for key in keys}

## What a result holds

The containers mirror the groups of a data file: `truth` (one entry per event), `response` (one row per event and one
column per sensor), `photons` (the photon bookkeeping per event), and the optional `hits` and `tracks`. Every
container returns its arrays by name with `arrays()`.

In [ ]:
for group in ("truth", "response", "photons"):
    print(group)
    for name, values in getattr(data, group).arrays().items():
        print(f"    {name:<22s} {values.dtype!s:<8s} {values.shape}")

The bookkeeping is exact: every emitted photon ends in exactly one way, and the charges of an event add up to its
detected photons plus its dark-noise hits (dark noise is off by default).

In [ ]:
p = data.photons
ended = p.n_detected + p.n_absorbed_bulk + p.n_absorbed_wall + p.n_absorbed_sensor + p.n_lost
print("every photon accounted for:", np.array_equal(ended, p.n_emitted))
print("charges add up:", np.array_equal(data.response.charge.sum(axis=1), p.n_detected + p.n_noise))

## Save and load

A file holds the data together with the configuration and the provenance; `describe` prints the same summary as
`flashsim info`.

In [ ]:
path = save(data, "../data/quickstart_notebook.h5")
print(describe(path))

In [ ]:
again = load(path)
print(again)
print("same configuration:", again.config == data.config)
print("same charges:", np.array_equal(again.response.charge, data.response.charge))

## One event

The response of an event is a charge and a mean hit time for every sensor. Here they are for the first event, against
the angle of the sensor on the wall. Proper event displays are task T4.

In [ ]:
i = 0
phi = np.degrees(data.detector.sensor_phi)
charge, tau = data.response.charge[i], data.response.time_mean[i]
x0, y0, energy = data.truth.x[i], data.truth.y[i], data.truth.energy[i]

fig, (top, bottom) = plt.subplots(2, 1, figsize=(6.4, 4.4), sharex=True, constrained_layout=True)
top.bar(phi, charge, width=8, color="#2a78d6")
top.set_ylabel("Charge [hits]")
top.set_title(f"Event {data.truth.event_id[i]}: vertex ({x0:.2f}, {y0:.2f}) m, E = {energy:.2f} MeV")
bottom.plot(phi, tau, "o", color="#eb6834")
bottom.set_ylabel("Mean hit time [ns]")
bottom.set_xlabel("Sensor angle [degrees]")
plt.show()

## The network input

The neural networks of the next stage see the log charges and the charge-weighted relative mean times of every event,
an array of shape `[N, 2M]` (definition in [docs/physics.md](../docs/physics.md#network-input)).

In [ ]:
x = network_input(data.response.charge, data.response.time_mean)
m = data.n_sensors
print("shape:", x.shape)
print("log charges, event 0, sensors 0-5:   ", np.round(x[0, :6], 2))
print("relative times, event 0, sensors 0-5:", np.round(x[0, m : m + 6], 2))

## Hits and photon paths

With `save_hits=True` every hit is stored with its sensor, its time and the history of its photon. With `save_tracks`
the photon paths are stored as polylines whose points carry a type: emission, scattering, reflection and the fate.

In [ ]:
hits = pd.DataFrame(data.hits.arrays())
hits[hits.event_id == 0].head()

In [ ]:
tracks = data.tracks
first = tracks.of_event(0)
print(len(tracks), "tracks stored,", len(first), "of them from event 0")
_, _, kind = tracks.polyline(first[0])
print("points of the first track:", [Interaction(k).name for k in kind])
fates, counts = np.unique(tracks.fate[first], return_counts=True)
print("fates of the tracks of event 0:", {Interaction(f).name: int(c) for f, c in zip(fates, counts, strict=True)})

## Next steps

- The tasks for the next session are in [TASKS.md](../TASKS.md): T0 to T4, and the bonus T5.
- Larger data sets are easiest to make on the command line, e.g. `flashsim generate -n 50000 --seed 5 -o
  data/t2_full.h5`; on a cluster, see [examples/cluster](../examples/cluster/README.md).